In [18]:
!pip install simpy

In [19]:
import simpy
import random
import statistics
from dataclasses import dataclass
from typing import Dict, List, Tuple
import time

@dataclass
class SimulationConfig:
    prep_capacity: int = 3
    rec_capacity: int = 3
    mean_interarrival: float = 25
    mean_prep: float = 40
    mean_surgery: float = 20
    mean_recovery: float = 40
    sim_time: float = 5000
    random_seed: int = None  # Make seed optional

class EnhancedSimulationMonitor:
    def __init__(self):
        self.arrival_times: Dict[str, float] = {}
        self.throughput_times: List[float] = []
        self.cumulative_blocking_time = 0.0
        self.prep_queue_lengths: List[Tuple[float, int]] = []
        self.surgery_queue_lengths: List[Tuple[float, int]] = []
        self.recovery_queue_lengths: List[Tuple[float, int]] = []
        self.ot_states: List[Tuple[float, str, float]] = []  # (start_time, state, duration)
        self.current_ot_state = "IDLE"
        self.ot_state_start = 0.0
        self.blocking_events = []

    def record_arrival(self, patient_id: str, time: float):
        self.arrival_times[patient_id] = time

    def record_departure(self, patient_id: str, time: float):
        if patient_id in self.arrival_times:
            throughput = time - self.arrival_times[patient_id]
            self.throughput_times.append(throughput)

    def add_blocking_time(self, duration: float, start_time: float):
        self.cumulative_blocking_time += duration
        self.blocking_events.append((start_time, duration))

    def record_queue_lengths(self, time: float, prep_q: int, surgery_q: int, recovery_q: int):
        self.prep_queue_lengths.append((time, prep_q))
        self.surgery_queue_lengths.append((time, surgery_q))
        self.recovery_queue_lengths.append((time, recovery_q))

    def change_ot_state(self, new_state: str, time: float):
        if self.current_ot_state != new_state:
            # Record duration of previous state
            if self.ot_state_start is not None:
                duration = time - self.ot_state_start
                self.ot_states.append((self.ot_state_start, self.current_ot_state, duration))
            self.current_ot_state = new_state
            self.ot_state_start = time

    def finalize(self, end_time: float):
        #state tracking at simulation end
        if self.ot_state_start is not None:
            duration = end_time - self.ot_state_start
            self.ot_states.append((self.ot_state_start, self.current_ot_state, duration))

    def get_metrics(self, sim_time: float):
        # Basic metrics
        avg_throughput = statistics.mean(self.throughput_times) if self.throughput_times else 0
        blocking_prob = self.cumulative_blocking_time / sim_time

        # Queue length calculations
        def calc_avg_queue(queue_data):
            if not queue_data:
                return 0
            total_weighted = 0
            last_time, last_length = queue_data[0]
            for time, length in queue_data[1:]:
                duration = time - last_time
                total_weighted += last_length * duration
                last_time, last_length = time, length
            # Add the last stage
            if last_time < sim_time:
                total_weighted += last_length * (sim_time - last_time)
            return total_weighted / sim_time

        avg_prep_queue = calc_avg_queue(self.prep_queue_lengths)
        avg_surgery_queue = calc_avg_queue(self.surgery_queue_lengths)
        avg_recovery_queue = calc_avg_queue(self.recovery_queue_lengths)

        # analysis OT state
        ot_utilization = 0
        ot_blocking = 0
        total_ot_time = 0

        for start, state, duration in self.ot_states:
            total_ot_time += duration
            if state == "BUSY":
                ot_utilization += duration
            elif state == "BLOCKED":
                ot_blocking += duration
                ot_utilization += duration  # OT is still utilized when blocked

        ot_utilization /= sim_time if sim_time > 0 else 1
        measured_blocking_prob = ot_blocking / sim_time if sim_time > 0 else 0

        return {
            'completed_patients': len(self.throughput_times),
            'avg_throughput_time': avg_throughput,
            'total_blocking_time': self.cumulative_blocking_time,
            'blocking_probability': blocking_prob,
            'measured_blocking_probability': measured_blocking_prob,
            'avg_prep_queue_length': avg_prep_queue,
            'avg_surgery_queue_length': avg_surgery_queue,
            'avg_recovery_queue_length': avg_recovery_queue,
            'ot_utilization': ot_utilization,
            'throughput_std_dev': statistics.stdev(self.throughput_times) if len(self.throughput_times) > 1 else 0
        }

def patient(env, config: SimulationConfig, monitor: EnhancedSimulationMonitor,
            prep_res, ot_res, rec_res, patient_id: str):

    monitor.record_arrival(patient_id, env.now)

    # Generate personal service times
    prep_time = random.expovariate(1 / config.mean_prep)
    surgery_time = random.expovariate(1 / config.mean_surgery)
    recovery_time = random.expovariate(1 / config.mean_recovery)

    # Preparation Phase
    with prep_res.request() as req:
        yield req
        yield env.timeout(prep_time)

    # Surgery Phase
    with ot_res.request() as ot_req:
        yield ot_req
        monitor.change_ot_state("BUSY", env.now)

        # Perform surgery
        yield env.timeout(surgery_time)

        # Critical blocking point - try to get recovery bed
        recovery_request = rec_res.request()
        request_time = env.now

        # Try immediate acquisition
        result = yield recovery_request | env.timeout(0)

        if recovery_request not in result:
            # Blocking occurs - cannot release patient from OT
            monitor.change_ot_state("BLOCKED", env.now)
            blocking_start = env.now
            yield recovery_request  # Wait for recovery bed
            blocking_duration = env.now - blocking_start
            monitor.add_blocking_time(blocking_duration, blocking_start)
            monitor.change_ot_state("BUSY", env.now)

        # Release OT (patient moves to recovery)
        # After the context manager exits, OT state will be updated

    # OT is released, update state to IDLE if no one is waiting.
    if len(ot_res.queue) == 0:
        monitor.change_ot_state("IDLE", env.now)

    # Recovery Phase
    yield env.timeout(recovery_time)
    rec_res.release(recovery_request)

    # Discharge
    monitor.record_departure(patient_id, env.now)

def patient_generator(env, config: SimulationConfig, monitor: EnhancedSimulationMonitor,
                     prep_res, ot_res, rec_res):
    patient_count = 0
    while True:
        patient_count += 1
        patient_id = f"Patient_{patient_count}"
        env.process(patient(env, config, monitor, prep_res, ot_res, rec_res, patient_id))
        interarrival = random.expovariate(1 / config.mean_interarrival)
        yield env.timeout(interarrival)

def enhanced_monitor(env, config: SimulationConfig, monitor: EnhancedSimulationMonitor,
                    prep_res, ot_res, rec_res, interval=5):
    while True:
        monitor.record_queue_lengths(
            env.now,
            len(prep_res.queue),
            len(ot_res.queue),
            len(rec_res.queue)
        )
        yield env.timeout(interval)

def run_enhanced_simulation(config: SimulationConfig = None):
    if config is None:
        config = SimulationConfig()

    # Use current time as seed if no seed provided, ensuring different results
    if config.random_seed is None:
        config.random_seed = int(time.time() * 1000) % 1000000

    random.seed(config.random_seed)
    env = simpy.Environment()
    monitor = EnhancedSimulationMonitor()

    prep_res = simpy.Resource(env, config.prep_capacity)
    ot_res = simpy.Resource(env, 1)
    rec_res = simpy.Resource(env, config.rec_capacity)

    env.process(patient_generator(env, config, monitor, prep_res, ot_res, rec_res))
    env.process(enhanced_monitor(env, config, monitor, prep_res, ot_res, rec_res))

    # Initialize OT state
    monitor.change_ot_state("IDLE", 0.0)

    env.run(until=config.sim_time)

    # Finalize monitoring
    monitor.finalize(config.sim_time)

    metrics = monitor.get_metrics(config.sim_time)

    print(f"\n{'='*80}")
    print(f"ENHANCED SIMULATION RESULTS (Seed: {config.random_seed})")
    print(f"{'='*80}")
    print(f"Completed patients: {metrics['completed_patients']}")
    print(f"Average throughput time: {metrics['avg_throughput_time']:.2f} ± {metrics['throughput_std_dev']:.2f}")
    print(f"Total OT blocking time: {metrics['total_blocking_time']:.2f}")
    print(f"Blocking probability: {metrics['blocking_probability']:.3f}")
    print(f"Measured blocking probability: {metrics['measured_blocking_probability']:.3f}")
    print(f"OT utilization: {metrics['ot_utilization']:.3f}")
    print(f"Average preparation queue: {metrics['avg_prep_queue_length']:.2f}")
    print(f"Average surgery queue: {metrics['avg_surgery_queue_length']:.2f}")
    print(f"Average recovery queue: {metrics['avg_recovery_queue_length']:.2f}")
    print(f"{'='*80}")

    return metrics, monitor

def run_comparative_analysis():
    scenarios = {
        "Baseline": SimulationConfig(random_seed=None),
        "High Load": SimulationConfig(mean_interarrival=20, random_seed=None),
        "Limited Recovery": SimulationConfig(rec_capacity=2, random_seed=None),
        "High Capacity": SimulationConfig(prep_capacity=4, rec_capacity=4, random_seed=None)
    }

    print(f"\n{'='*80}")
    print("COMPARATIVE ANALYSIS")
    print(f"{'='*80}")
    print(f"{'Scenario':<15} {'Throughput':<12} {'Blocking %':<12} {'OT Util':<10} {'Prep Queue':<12} {'Surg Queue':<12}")
    print(f"{'-'*80}")

    results_summary = []

    for name, config in scenarios.items():
        print(f"Running {name}...")
        results, _ = run_enhanced_simulation(config)
        results_summary.append((name, results))

        print(f"{name:<15} {results['avg_throughput_time']:>10.1f} {results['blocking_probability']:>11.3f} "
              f"{results['ot_utilization']:>9.3f} {results['avg_prep_queue_length']:>11.2f} "
              f"{results['avg_surgery_queue_length']:>11.2f}")

    return results_summary

if __name__ == "__main__":
    # Get different results on each test
    print("Testing randomization - running baseline twice:")
    print("\nFirst run:")
    config1 = SimulationConfig(random_seed=None)
    results1, _ = run_enhanced_simulation(config1)

    print("\nSecond run (different result):")
    config2 = SimulationConfig(random_seed=None)
    results2, _ = run_enhanced_simulation(config2)

    # Verify they're different
    if abs(results1['avg_throughput_time'] - results2['avg_throughput_time']) > 1.0:
        print("\n Success: Results are different between runs!")
    else:
        print("\n Warning: Results are very similar - may need stronger randomization")

    # Comparative analysis
    run_comparative_analysis()

Testing randomization - running baseline twice:

First run:

ENHANCED SIMULATION RESULTS (Seed: 868422)
Completed patients: 205
Average throughput time: 312.89 ± 115.58
Total OT blocking time: 700.38
Blocking probability: 0.140
Measured blocking probability: 0.146
OT utilization: 0.969
Average preparation queue: 0.33
Average surgery queue: 8.26
Average recovery queue: 0.15

Second run (different result):

ENHANCED SIMULATION RESULTS (Seed: 868438)
Completed patients: 202
Average throughput time: 208.60 ± 134.33
Total OT blocking time: 501.03
Blocking probability: 0.100
Measured blocking probability: 0.100
OT utilization: 0.894
Average preparation queue: 0.41
Average surgery queue: 4.95
Average recovery queue: 0.10

 Success: Results are different between runs!

COMPARATIVE ANALYSIS
Scenario        Throughput   Blocking %   OT Util    Prep Queue   Surg Queue  
--------------------------------------------------------------------------------
Running Baseline...

ENHANCED SIMULATION RESULT